In [ ]:
import json
import hashlib
from pathlib import Path
from datetime import date, timedelta
from getpass import getpass

import requests

BASE_URL = "https://api.sectors.app"
OUT = Path("/content/sectors_lq45")
DAILY_DIR = OUT / "daily"
MAX_REQUESTS = 300

API_KEY = getpass("Masukkan Sectors API key: ")
if not API_KEY.strip():
    raise ValueError("API key kosong")

APPROVAL = input("Ketik APPROVE SECTORS API CALL untuk lanjut: ")
if APPROVAL != "APPROVE SECTORS API CALL":
    raise RuntimeError("Request dihentikan")

session = requests.Session()
session.headers.update({
    "Authorization": API_KEY.strip(),
    "Accept": "application/json",
    "User-Agent": "IDX-Evidence-Lab-Colab/1.0",
})

used_requests = 0
DAILY_DIR.mkdir(parents=True, exist_ok=True)

def request_json(path, params=None):
    global used_requests

    if used_requests >= MAX_REQUESTS:
        raise RuntimeError("Batas request batch tercapai")

    response = session.get(
        BASE_URL + path,
        params=params,
        timeout=60,
    )
    used_requests += 1
    response.raise_for_status()

    raw = response.content
    payload = response.json()

    metadata = {
        "provider": "Sectors.app",
        "endpoint": response.url,
        "http_status": response.status_code,
        "retrieved_at": date.today().isoformat(),
        "sha256": hashlib.sha256(raw).hexdigest(),
        "request_number": used_requests,
    }

    return payload, metadata

# 1. Ambil universe sekali
universe_path = OUT / "lq45-universe.json"

if universe_path.exists():
    universe = json.loads(universe_path.read_text())
else:
    universe, metadata = request_json(
        "/v2/companies/",
        params={
            "where": "indices in ['LQ45']",
            "order_by": "symbol",
            "limit": 100,
            "offset": 0,
            "include_query_values": "true",
        },
    )

    universe_path.write_text(
        json.dumps(universe, ensure_ascii=False, indent=2)
    )

    (OUT / "lq45-universe.metadata.json").write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2)
    )

# 2. Normalisasi ticker
rows = universe.get("results", [])
tickers = []

for row in rows:
    ticker = row.get("symbol") or row.get("ticker")
    if ticker:
        ticker = ticker.replace(".JK", "")
        tickers.append(ticker)

tickers = sorted(set(tickers))

if len(tickers) != 45:
    print(f"Peringatan: jumlah ticker terbaca {len(tickers)}, bukan 45")

# 3. Rentang tiga tahun
END_DATE = date.today()
START_DATE = END_DATE - timedelta(days=3 * 365)

def windows(start, end, max_days=90):
    current = start

    while current <= end:
        window_end = min(
            current + timedelta(days=max_days - 1),
            end
        )
        yield current, window_end
        current = window_end + timedelta(days=1)

# 4. Ambil hanya window yang belum tersedia
for ticker in tickers:
    ticker_dir = DAILY_DIR / ticker
    ticker_dir.mkdir(parents=True, exist_ok=True)

    for start, end in windows(START_DATE, END_DATE):
        stem = f"{ticker}_{start}_{end}"
        data_path = ticker_dir / f"{stem}.json"
        metadata_path = ticker_dir / f"{stem}.metadata.json"

        if data_path.exists() and metadata_path.exists():
            print("SKIP", stem)
            continue

        if used_requests >= MAX_REQUESTS:
            print("BATCH LIMIT tercapai. Jalankan ulang notebook untuk resume.")
            raise SystemExit

        payload, metadata = request_json(
            f"/v2/daily/{ticker}/",
            params={
                "start": start.isoformat(),
                "end": end.isoformat(),
            },
        )

        data_path.write_text(
            json.dumps(payload, ensure_ascii=False, indent=2)
        )
        metadata_path.write_text(
            json.dumps(metadata, ensure_ascii=False, indent=2)
        )

        print("SAVED", stem)

print(f"Selesai. Request batch ini: {used_requests}/{MAX_REQUESTS}")